# Starling IDP Ensemble Generation

Generates a conformational ensemble for an intrinsically disordered protein (IDP) region using [Starling](https://github.com/idptools/starling).

**Requires**: GPU runtime (`Runtime > Change runtime type > T4 GPU`)

## What this notebook does
1. Installs Starling + MDAnalysis
2. Runs Starling on a trimmed IDP sequence
3. Extracts individual PDB frames from the XTC trajectory
4. Downloads results as a zip

## Input
- A trimmed IDP sequence (no structured domains — determine boundaries via InterPro first)
- Default example: Tau K18 repeat domain (MTBR, residues 244–368, 2N4R)


## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch to GPU runtime')

## 1. Install Starling

In [ ]:
%%capture
!pip install idptools-starling MDAnalysis idptools-afrc

In [ ]:
# Verify install
!starling --help 2>&1 | head -5

## 2. Define your IDP sequence

Paste your trimmed IDP sequence below. **Do not include structured domains** — use InterPro to find the boundary first.

Sequence must be:
- Single-letter amino acid code, no spaces
- Genuinely disordered region only
- Typically 50–300 aa (longer = slower)

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────

PROTEIN_NAME = "tau_k18"          # used for output filenames
N_CONFORMERS = 100                # 100 for exploration, 400 for production

# Tau K18 repeat domain (MTBR, residues 244-368, 2N4R numbering)
# Replace with your sequence — keep it to the IDP region only
SEQUENCE = "KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYKPVDLSKVTSKCGSLGNIHHKPGGGQVEVKSEKLDFKDRVQSKIGSLDNITHVPGGGNKKIETHKLTFRENAKAKTDHGAEIVYKSPVVSGDTSPRHLSNVSSTGSIDMVDSPQLATLADEVSASLAKQGL"

# ────────────────────────────────────────────────────────────────────────────

print(f"Protein  : {PROTEIN_NAME}")
print(f"Length   : {len(SEQUENCE)} aa")
print(f"Conformers: {N_CONFORMERS}")
print(f"Sequence : {SEQUENCE[:40]}...")

## 3. Run Starling

Outputs:
- `{PROTEIN_NAME}_STARLING.pdb` — topology file
- `{PROTEIN_NAME}_STARLING.xtc` — trajectory (N_CONFORMERS frames)

In [ ]:
import os
os.makedirs('starling_output', exist_ok=True)
%cd starling_output

In [ ]:
cmd = f"starling {SEQUENCE} --outname {PROTEIN_NAME} -c {N_CONFORMERS} -r"
print(f"Running: {cmd}\n")
!{cmd}

In [ ]:
import os
pdb_file = f"{PROTEIN_NAME}_STARLING.pdb"
xtc_file = f"{PROTEIN_NAME}_STARLING.xtc"

assert os.path.exists(pdb_file), f"PDB not found: {pdb_file}"
assert os.path.exists(xtc_file), f"XTC not found: {xtc_file}"

print(f"PDB : {pdb_file}  ({os.path.getsize(pdb_file)//1024} KB)")
print(f"XTC : {xtc_file}  ({os.path.getsize(xtc_file)//1024} KB)")

## 4. Extract individual PDB frames

In [ ]:
import MDAnalysis as mda
import numpy as np
import os

def extract_frames(topology, trajectory, output_dir, n_frames=100):
    os.makedirs(output_dir, exist_ok=True)
    u = mda.Universe(topology, trajectory)
    total = len(u.trajectory)
    print(f"Trajectory: {total} frames total")

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    saved = []
    for i, ts_idx in enumerate(indices):
        u.trajectory[ts_idx]
        out = os.path.join(output_dir, f"frame_{i+1:04d}.pdb")
        u.atoms.write(out)
        saved.append(out)

    print(f"Saved {len(saved)} frames to {output_dir}/")
    return saved


frames_dir = f"{PROTEIN_NAME}_frames"
frames = extract_frames(pdb_file, xtc_file, frames_dir, n_frames=100)
print(f"\nFirst frame: {frames[0]}")

## 5. Quick ensemble analysis

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import numpy as np
import matplotlib.pyplot as plt

u = mda.Universe(pdb_file, xtc_file)
ca = u.select_atoms('name CA')

# Rg across trajectory
rg_values = []
for ts in u.trajectory:
    rg_values.append(ca.radius_of_gyration())

rg = np.array(rg_values)
print(f"Rg  mean={rg.mean():.1f} Å   std={rg.std():.1f} Å   min={rg.min():.1f}   max={rg.max():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rg, linewidth=0.8, color='steelblue')
axes[0].set_xlabel('Conformer index')
axes[0].set_ylabel('Radius of gyration (Å)')
axes[0].set_title(f'{PROTEIN_NAME} — Rg across ensemble')
axes[0].axhline(rg.mean(), color='red', linestyle='--', linewidth=1, label=f'mean={rg.mean():.1f} Å')
axes[0].legend()

axes[1].hist(rg, bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Radius of gyration (Å)')
axes[1].set_ylabel('Count')
axes[1].set_title('Rg distribution')

plt.tight_layout()
plt.savefig(f'{PROTEIN_NAME}_rg_analysis.png', dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
# End-to-end distance (first CA to last CA) — another disorder metric
u.trajectory[0]
ca_atoms = u.select_atoms('name CA')

ete_values = []
for ts in u.trajectory:
    first = ca_atoms.positions[0]
    last  = ca_atoms.positions[-1]
    ete_values.append(np.linalg.norm(last - first))

ete = np.array(ete_values)
print(f"End-to-end dist  mean={ete.mean():.1f} Å   std={ete.std():.1f} Å   min={ete.min():.1f}   max={ete.max():.1f}")

plt.figure(figsize=(6, 4))
plt.hist(ete, bins=30, color='coral', edgecolor='white')
plt.xlabel('End-to-end distance (Å)')
plt.ylabel('Count')
plt.title(f'{PROTEIN_NAME} — End-to-end distance distribution')
plt.tight_layout()
plt.savefig(f'{PROTEIN_NAME}_ete_analysis.png', dpi=150)
plt.show()

## 5b. Conformer Filtering + Structural Analysis

Three analyses for early conformer trimming and ensemble characterization:
1. **Rg-based filtering** — cut outliers outside ±2σ, compare to Flory scaling prediction
2. **Pairwise Cα distance distribution** — all residue-residue distances across ensemble
3. **Inner-residue scaling plot** — mean distance vs sequence separation |i−j| (Flory exponent fit)
4. **Mean distance map** — N×N heatmap of average Cα–Cα distances across ensemble

In [ ]:
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import os

# ── Load AFRC — sequence-specific random coil reference state ────────────────
from afrc import AnalyticalFRC
afrc        = AnalyticalFRC(SEQUENCE)
rg_afrc     = afrc.get_mean_radius_of_gyration()       # sequence-aware Rg
re_afrc     = afrc.get_mean_end_to_end_distance()      # sequence-aware Re
afrc_dist   = afrc.get_distance_map(symmetric_map=True)  # (N×N) R(i,j) matrix

print(f'AFRC reference ({PROTEIN_NAME}, {len(SEQUENCE)} aa)')
print(f'  Rg : {rg_afrc:.1f} Å')
print(f'  Re : {re_afrc:.1f} Å')

# ── Load trajectory ───────────────────────────────────────────────────────────
u             = mda.Universe(pdb_file, xtc_file)
ca            = u.select_atoms('name CA')
n_residues    = len(ca)
n_frames_traj = len(u.trajectory)
print(f'  Residues : {n_residues}  |  Frames : {n_frames_traj}')

# ── 1. Rg filter (±2σ window, anchored to AFRC reference) ────────────────────
rg_all          = np.array([ca.radius_of_gyration() for ts in u.trajectory])
rg_mean, rg_std = rg_all.mean(), rg_all.std()
rg_lo, rg_hi    = rg_mean - 2*rg_std, rg_mean + 2*rg_std
keep_idx        = np.where((rg_all >= rg_lo) & (rg_all <= rg_hi))[0]

print(f'\nRg observed : mean={rg_mean:.1f}  std={rg_std:.1f} Å')
print(f'AFRC ref    : {rg_afrc:.1f} Å  (sequence-specific)')
print(f'Deviation   : {rg_mean - rg_afrc:+.1f} Å  ({100*(rg_mean-rg_afrc)/rg_afrc:+.1f}%)')
print(f'Keep window : {rg_lo:.1f} – {rg_hi:.1f} Å')
print(f'Passing     : {len(keep_idx)}/{n_frames_traj}  ({100*len(keep_idx)/n_frames_traj:.0f}%)')

# ── 2. Collect Cα positions for passing frames ────────────────────────────────
positions = []
for idx in keep_idx:
    u.trajectory[idx]
    positions.append(ca.positions.copy())
positions = np.array(positions)   # (n_keep, n_residues, 3)

# ── 3. Observed mean Cα distance map ─────────────────────────────────────────
n_keep   = len(positions)
obs_dist = np.zeros((n_residues, n_residues))
for k in range(n_keep):
    diff = positions[k][:, None, :] - positions[k][None, :, :]  # (N,N,3)
    obs_dist += np.linalg.norm(diff, axis=-1)
obs_dist /= n_keep

# ── 4. Fractional deviation from AFRC R(i,j) ─────────────────────────────────
# Trim afrc_dist to match ensemble residue count (in case of size mismatch)
afrc_dist = afrc_dist[:n_residues, :n_residues]

with np.errstate(invalid='ignore', divide='ignore'):
    deviation = np.where(afrc_dist > 0,
                         (obs_dist - afrc_dist) / afrc_dist,
                         0.0)

# Per-residue marginal deviation — trim guide
np.fill_diagonal(deviation, 0)
marginal_dev = np.abs(deviation).mean(axis=1)

# ── 5. Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Rg distribution with AFRC reference line
ax = axes[0, 0]
ax.hist(rg_all, bins=30, color='steelblue', edgecolor='white', alpha=0.8, label='All conformers')
ax.hist(rg_all[keep_idx], bins=30, color='seagreen', edgecolor='white', alpha=0.6, label='Passing (±2σ)')
ax.axvline(rg_mean, color='steelblue', linestyle='--', lw=1.5, label=f'Observed mean = {rg_mean:.1f} Å')
ax.axvline(rg_afrc, color='tomato',    linestyle='-',  lw=2,   label=f'AFRC ref = {rg_afrc:.1f} Å')
ax.axvline(rg_lo,   color='gray',      linestyle=':',  lw=1)
ax.axvline(rg_hi,   color='gray',      linestyle=':',  lw=1, label='±2σ window')
ax.set_xlabel('Rg (Å)'); ax.set_ylabel('Count')
ax.set_title('Rg distribution vs AFRC reference')
ax.legend(fontsize=7)

# B: Per-residue marginal deviation — trim guide
ax = axes[0, 1]
ax.plot(range(1, n_residues+1), marginal_dev, color='darkorange', lw=1.5)
ax.axhline(marginal_dev.mean(), color='gray', linestyle='--', lw=1,
           label=f'mean = {marginal_dev.mean():.3f}')
ax.axvspan(2,  7,  alpha=0.2, color='royalblue', label='PHF6* (2–7)')
ax.axvspan(33, 38, alpha=0.2, color='crimson',   label='PHF6 (33–38)')
ax.set_xlabel('Residue position (1-indexed)')
ax.set_ylabel('Mean |deviation| from AFRC R(i,j)')
ax.set_title('Per-residue deviation from random coil\n'
             'Peaks at termini = trim candidates')
ax.legend(fontsize=7)

# C: Observed mean Cα distance map
ax = axes[1, 0]
im = ax.imshow(obs_dist, cmap='viridis_r', aspect='auto')
plt.colorbar(im, ax=ax, label='Mean Cα–Cα distance (Å)')
ax.set_xlabel('Residue j'); ax.set_ylabel('Residue i')
ax.set_title('Observed mean Cα–Cα distance map')

# D: Fractional deviation from AFRC — trim + contact guide
ax = axes[1, 1]
vmax = max(0.3, float(np.abs(deviation).max()) * 0.8)
im2  = ax.imshow(deviation, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
plt.colorbar(im2, ax=ax, label='(obs − AFRC) / AFRC')
ax.set_xlabel('Residue j'); ax.set_ylabel('Residue i')
ax.set_title('Fractional deviation from AFRC R(i,j)\n'
             'Red = extended, Blue = compact vs random coil')

plt.suptitle(f'{PROTEIN_NAME} — Ensemble vs AFRC reference state', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{PROTEIN_NAME}_ensemble_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved ensemble_analysis.png')
print('\n→ Plot B: terminal residues with high deviation = trim candidates')
print('→ Plot D: blue off-diagonal blocks = transient contacts (aggregation-prone regions)')


In [ ]:
# Write filtered frames to a separate directory → these go to FoldMason
import shutil

filtered_dir = f"{PROTEIN_NAME}_frames_filtered"
os.makedirs(filtered_dir, exist_ok=True)

for rank, frame_idx in enumerate(keep_idx):
    src = os.path.join(frames_dir, f"frame_{frame_idx+1:04d}.pdb")
    dst = os.path.join(filtered_dir, f"filtered_{rank+1:04d}.pdb")
    if os.path.exists(src):
        shutil.copy2(src, dst)

print(f"Filtered frames written to: {filtered_dir}/")
print(f"Count: {len(keep_idx)} — these are ready for FoldMason")
print(f"\nDropped {n_frames_traj - len(keep_idx)} outlier conformers ({100*(n_frames_traj-len(keep_idx))/n_frames_traj:.1f}%)")

## 6. Download results

In [ ]:
import shutil
from google.colab import files

zip_name = f"{PROTEIN_NAME}_starling_results"
shutil.make_archive(zip_name, 'zip', '.', frames_dir)

import zipfile
with zipfile.ZipFile(f"{zip_name}.zip", 'a') as zf:
    # Raw frames
    zf.write(pdb_file)
    zf.write(xtc_file)
    # Filtered frames (FoldMason-ready)
    for f in sorted(os.listdir(filtered_dir)):
        zf.write(os.path.join(filtered_dir, f))
    # Plots
    for png in [
        f'{PROTEIN_NAME}_rg_analysis.png',
        f'{PROTEIN_NAME}_ete_analysis.png',
        f'{PROTEIN_NAME}_ensemble_analysis.png',
    ]:
        if os.path.exists(png):
            zf.write(png)

print(f"Downloading {zip_name}.zip")
files.download(f"{zip_name}.zip")

## Notes

- **Rg** (radius of gyration): measures compactness. A disordered chain should show a wide distribution — large variance = genuinely disordered. A narrow peak at low Rg = collapsed/structured-like.
- **End-to-end distance**: complementary measure of chain extension. IDPs should span a wide range.
- **Next steps**: upload the `_frames/` PDB files to FoldMason for ensemble QC, or use them directly as Starling-based IDP conformers for binder design.
- **For production**: increase `N_CONFORMERS` to 400 and `n_frames` to 100.
